# Debug Realingo Scraper for Praha Rentals
This notebook reproduces the failed Realingo request, inspects the response, updates the URL and parameters, adds error handling, and verifies listing iteration.

In [ ]:
## 1. Import Required Libraries
Import httpx, logging, and parsing utilities for the scraper.
</VSCode.Cell>
<VSCode.Cell id="code-imports" language="python">
import logging
import re
from typing import Optional

import httpx
</VSCode.Cell>
<VSCode.Cell id="reproduce-cell" language="markdown">
## 2. Reproduce the Failed Request
Send the same GET request to Realingo and reproduce the 404 error using httpx.
</VSCode.Cell>
<VSCode.Cell id="code-reproduce" language="python">
BASE_URL = 'https://www.realingo.cz'
SEARCH_URL = f'{BASE_URL}/vyhledavani'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
}
params = {'city': 'Praha', 'offerType': 'pronajem', 'page': 1}

with httpx.Client() as client:
    resp = client.get(SEARCH_URL, params=params, headers=headers, timeout=30.0)
    print('url:', resp.url)
    print('status code:', resp.status_code)
    print('headers:', dict(resp.headers))
    print('body snippet:', resp.text[:1000])
</VSCode.Cell>
<VSCode.Cell id="inspect-cell" language="markdown">
## 3. Inspect Response Status and Content
Check whether the 404 occurs because the endpoint or parameters are wrong, or because the site requires a different URL.
</VSCode.Cell>
<VSCode.Cell id="code-inspect" language="python">
def inspect_homepage():
    with httpx.Client() as client:
        r = client.get(BASE_URL, headers=headers, timeout=30.0)
        print('homepage status:', r.status_code)
        print('homepage url:', r.url)
        print('contains /vyhledavani:', '/vyhledavani' in r.text)
        print('contains /api:', '/api/' in r.text)
        print('contains /search:', '/search' in r.text)
        print('sample snippet:')
        print(r.text[:3000])

inspect_homepage()
</VSCode.Cell>
<VSCode.Cell id="update-url-cell" language="markdown">
## 4. Update Search URL and Query Parameters
Try different URLs and query parameters until the request succeeds.
</VSCode.Cell>
<VSCode.Cell id="code-update-url" language="python">
candidate_urls = [
    'https://www.realingo.cz/vyhledavani',
    'https://www.realingo.cz/vyhledavani/pronajem',
    'https://www.realingo.cz/byt-pronajem',
    'https://www.realingo.cz/poskytovane-nabidky',
    'https://www.realingo.cz/api/offers',
]

with httpx.Client() as client:
    for url in candidate_urls:
        try:
            resp = client.get(url, params=params, headers=headers, timeout=30.0)
            print('URL:', url, 'status:', resp.status_code)
        except Exception as e:
            print('URL:', url, 'error:', type(e).__name__, e)
</VSCode.Cell>
<VSCode.Cell id="error-handling-cell" language="markdown">
## 5. Add HTTP Error Handling
Wrap requests in try/except and log the error with response details.
</VSCode.Cell>
<VSCode.Cell id="code-error-handling" language="python">
def safe_get(client: httpx.Client, url: str, **kwargs) -> Optional[httpx.Response]:
    try:
        resp = client.get(url, **kwargs)
        resp.raise_for_status()
        return resp
    except httpx.HTTPStatusError as e:
        logging.error('HTTP error for %s: %s', url, e)
        if e.response is not None:
            logging.error('response status: %s', e.response.status_code)
            logging.error('response body: %s', e.response.text[:500])
    except Exception as e:
        logging.error('Request failed for %s: %s', url, e)
    return None

with httpx.Client() as client:
    response = safe_get(client, SEARCH_URL, params=params, headers=headers, timeout=30.0)
    print('safe_get response:', response)
</VSCode.Cell>
<VSCode.Cell id="test-iteration-cell" language="markdown">
## 6. Test the Scraper Iteration Logic
Run the page iteration loop and validate that listings are returned without HTTP errors.
</VSCode.Cell>
<VSCode.Cell id="code-test-iteration" language="python">
def find_listings(html: str):
    matches = re.findall(r'<a[^>]+href=[\\"\\']([^\\"\\']+)[\\"\\']', html, re.IGNORECASE)
    print('found hrefs count:', len(matches))
    print(matches[:20])

with httpx.Client() as client:
    resp = safe_get(client, SEARCH_URL, params=params, headers=headers, timeout=30.0)
    if resp is not None:
        find_listings(resp.text)
